# OCR Pipeline for Historical Print Periodicals

Automated full-text OCR for scanned periodicals using the Mistral OCR API (`mistral-ocr-latest`).

**Features:** Structured extraction (headings, paragraphs, footnotes) | Auto-splitting of large PDFs (>50 MB) | SQLite checkpoint system | Output as Markdown, Plain Text, and JSON

**Workflow:** Installation → Setup → Discovery → Batch OCR → Cleanup

**Documentation:** `docs/ARCHITECTURE.md` (System architecture) | `docs/LLM_WORKFLOW.md` (API workflow)

---

## Prerequisites

**Mistral API Key required:** Registration at [console.mistral.ai](https://console.mistral.ai) → API Keys → Create new key

**Configuration:** Add your API key to the `.env` file (project root):
```
MISTRAL_API_KEY=your_mistral_api_key_here
```

**Installation:** The following cell installs all Python packages automatically

---

## Cell 1: Setup and Imports

Loads Mistral AI SDK and local helper functions, configures logging, creates project directories (`data/input`, `data/output`, `data/tracking`), and initializes the SQLite database for the checkpoint system.

**Config via `.env`:** `MISTRAL_MODEL` (default: mistral-ocr-latest) | `DELAY_SECONDS` (rate limiting) | `MAX_RETRIES` | `TIMEOUT_SECONDS`

**Output:** Status messages with API key check, path overview, and configuration

In [ ]:
# Install dependencies (run once)
%pip install -r ../requirements.txt

# Imports
import os
import json
import logging
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from mistralai.client import Mistral

# Import local utils (explicit imports for better code readability)
from utils import (
    init_tracking_database,
    get_processed_pdfs,
    prepare_file_for_ocr,
    process_single_pdf,
    get_processing_stats,
    get_storage_stats,
    cleanup_temp_files
)

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Load API key
load_dotenv(Path("..") / ".env")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

if not MISTRAL_API_KEY:
    raise ValueError(
        "MISTRAL_API_KEY not found! "
        "Please create .env file in project root with: MISTRAL_API_KEY=your_key_here"
    )

# Initialize Mistral client
try:
    client = Mistral(api_key=MISTRAL_API_KEY)
    MISTRAL_MODEL = os.getenv("MISTRAL_MODEL", "mistral-ocr-latest")
except Exception as e:
    raise RuntimeError(
        f"Mistral client could not be initialized: {e}\n\n"
        "Possible causes:\n"
        "  - Mistral SDK not installed: pip install mistralai\n"
        "  - Invalid API key format\n"
        "  - Import error in mistralai package"
    ) from e

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_INPUT = PROJECT_ROOT / "data" / "input"
DATA_OUTPUT = PROJECT_ROOT / "data" / "output"
DATA_TRACKING = PROJECT_ROOT / "data" / "tracking"

# Create directories if they don't exist
DATA_INPUT.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT.mkdir(parents=True, exist_ok=True)
DATA_TRACKING.mkdir(parents=True, exist_ok=True)

# Configuration
DELAY_SECONDS = float(os.getenv("DELAY_SECONDS", "2.0"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "5"))
TIMEOUT = int(os.getenv("TIMEOUT_SECONDS", "120"))

# Tracking database
DB_PATH = str(DATA_TRACKING / "ocr_progress.db")
init_tracking_database(DB_PATH)

# Status output
print("="*70)
print("OCR PIPELINE FOR HISTORICAL PERIODICALS")
print("="*70)
print("✓ Mistral client initialized")
print(f"✓ Model: {MISTRAL_MODEL}")
print(f"✓ API Key: {MISTRAL_API_KEY[:8]}...{MISTRAL_API_KEY[-4:]}")
print(f"✓ Tracking DB: {DB_PATH}")
print("")
print(f"📁 Paths:")
print(f"  Input:    {DATA_INPUT}")
print(f"  Output:   {DATA_OUTPUT}")
print(f"  Tracking: {DATA_TRACKING}")
print("")
print("⚙️ Configuration:")
print(f"  Delay:      {DELAY_SECONDS}s")
print(f"  Max Retry:  {MAX_RETRIES}")
print(f"  Timeout:    {TIMEOUT}s")
print("="*70)
print(f"✓ Setup complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

Note: you may need to restart the kernel to use updated packages.


2026-03-16 12:33:42,029 - utils - INFO - Tracking-Datenbank initialisiert: /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/tracking/ocr_progress.db


OCR PIPELINE FOR HISTORICAL PERIODICALS
✓ Mistral client initialized
✓ Model: mistral-ocr-latest
✓ API Key: DDWQE5hS...SnJf
✓ Tracking DB: /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/tracking/ocr_progress.db

📁 Paths:
  Input:    /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/input
  Output:   /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/output
  Tracking: /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/tracking

⚙️ Configuration:
  Delay:      1.5s
  Max Retry:  5
  Timeout:    120s
✓ Setup complete: 2026-03-16 12:33:42


## Cell 2: File Discovery

Searches the input directory (`data/input/`) recursively for PDFs (`.pdf`) and images (`.jpg`, `.jpeg`, `.png`). Collects metadata (file path, type, size) and creates a list of all files to process.

**Output:** Statistics overview (number of PDFs, images, total size) and detailed list of all found files

**Note:** Large PDFs (>50 MB or >1000 pages) will be automatically split into 500-page chunks in Cell 3

In [ ]:
# Supported file formats
SUPPORTED_EXTENSIONS = {
    'pdf': ['.pdf'],
    'image': ['.jpg', '.jpeg', '.png']
}

# Find all files in input folder
all_files = []
skipped_files = []

for root, dirs, files in os.walk(DATA_INPUT):
    for filename in files:
        file_path = Path(root) / filename
        ext = file_path.suffix.lower()
        
        # Check if supported format
        file_type = None
        if ext in SUPPORTED_EXTENSIONS['pdf']:
            file_type = 'PDF'
        elif ext in SUPPORTED_EXTENSIONS['image']:
            file_type = 'Image'
        
        if file_type:
            try:
                # Get file size (can fail for symlinks, permission errors, etc.)
                size_mb = file_path.stat().st_size / (1024 * 1024)
                all_files.append({
                    'path': str(file_path),
                    'filename': filename,
                    'type': file_type,
                    'size_mb': round(size_mb, 2),
                    'extension': ext
                })
            except (OSError, PermissionError) as e:
                # File not accessible (symlink, permission, cloud sync, etc.)
                logger.warning(f"File skipped (no access): {filename} - {e}")
                skipped_files.append({'filename': filename, 'error': str(e)})
                continue

# Statistics
pdfs = [f for f in all_files if f['type'] == 'PDF']
images = [f for f in all_files if f['type'] == 'Image']
total_size = sum(f['size_mb'] for f in all_files)

# Output
print("="*70)
print("FILE DISCOVERY")
print("="*70)
print(f"📁 Input directory: {DATA_INPUT}")
print("")
print("📊 Statistics:")
print(f"  PDFs:   {len(pdfs)}")
print(f"  Images: {len(images)}")
print(f"  Total:  {len(all_files)} files")
print(f"  Size:   {total_size:.2f} MB")

if skipped_files:
    print(f"  ⚠️ Skipped: {len(skipped_files)} files (no access)")

print(f"")

if not all_files:
    print("⚠️  NO FILES FOUND!")
    print(f"   Please place PDFs or images in {DATA_INPUT}")
else:
    print("📄 Found files:")
    print("")
    for i, file_info in enumerate(all_files, 1):
        print(f"{i:3d}. [{file_info['type']:5s}] {file_info['filename']:50s} ({file_info['size_mb']:6.2f} MB)")

if skipped_files:
    print(f"")
    print(f"⚠️  Skipped files:")
    for skip in skipped_files[:5]:  # Show max 5
        print(f"  - {skip['filename']}: {skip['error']}")
    if len(skipped_files) > 5:
        print(f"  ... and {len(skipped_files) - 5} more")

print("="*70)

# Save list for further processing
FILES_TO_PROCESS = all_files

FILE DISCOVERY
📁 Input directory: /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/input

📊 Statistics:
  PDFs:   0
  Images: 20
  Total:  20 files
  Size:   25.67 MB

📄 Found files:

  1. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p008.jpg (  1.17 MB)
  2. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p010.jpg (  1.15 MB)
  3. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p020.jpg (  1.27 MB)
  4. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p009.jpg (  1.25 MB)
  5. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p019.jpg (  1.32 MB)
  6. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p007.jpg (  0.84 MB)
  7. [Image] TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20

## Cell 3: Batch OCR Processing

Processes all files found in Cell 2 with the Mistral OCR API. The checkpoint system automatically skips already-processed files (resumable after interruption). Large PDFs are split into chunks (→ `data/tracking/pdf_chunks/`), OCR processing includes retry logic for API errors.

**Output:** Live progress display per file | Chunk statistics | Final overall statistics (files, pages, API calls, costs)

**Results:** `.md` (Markdown with structure) | `.txt` (Plain text) | `_metadata.json` (Confidence, page count, warnings) + SQLite DB update

**Resume:** If interrupted, re-run this cell → Checkpoint system skips completed files

In [ ]:
# Temp directory for split PDFs
TEMP_DIR = DATA_TRACKING / "pdf_chunks"
TEMP_DIR.mkdir(exist_ok=True)

# Load already processed files
processed_files = get_processed_pdfs(DB_PATH)

# Statistics
total_files = len(FILES_TO_PROCESS)
total_processed = 0
total_errors = 0
total_api_calls = 0
start_time = datetime.now()

print("="*70)
print("BATCH OCR PROCESSING")
print("="*70)
print("📊 Status:")
print(f"  Total files:        {total_files}")
print(f"  Already processed:  {len(processed_files)}")
print(f"  To do:              {total_files - len(processed_files)}")
print("")
print("🚀 Starting processing...")
print("="*70)
print("")

# Loop over all files
for file_idx, file_info in enumerate(FILES_TO_PROCESS, 1):
    file_path = file_info['path']
    file_id = Path(file_path).stem

    # Skip if already processed
    if file_id in processed_files:
        print(f"[{file_idx}/{total_files}] ⏭️  SKIP: {file_info['filename']} (already processed)")
        continue

    print("")
    print("="*70)
    print(f"[{file_idx}/{total_files}] 📄 Processing: {file_info['filename']}")
    print(f"  Type:   {file_info['type']}")
    print(f"  Size:   {file_info['size_mb']} MB")
    print("="*70)

    try:
        # 1. Prepare file (split if needed)
        print("\n📦 Preparing file...")
        files_to_ocr = prepare_file_for_ocr(file_path, str(TEMP_DIR))

        if len(files_to_ocr) > 1:
            print(f"✓ PDF split: {len(files_to_ocr)} chunks")
        else:
            print("✓ File ready (no splitting needed)")

        # 2. Process all chunks
        all_results = []

        for chunk_idx, chunk_path in enumerate(files_to_ocr, 1):
            chunk_name = Path(chunk_path).name

            if len(files_to_ocr) > 1:
                print(f"\n  [{chunk_idx}/{len(files_to_ocr)}] 🔄 Processing chunk: {chunk_name}")
            else:
                print("\n🔄 Starting OCR...")

            # OCR processing
            result = process_single_pdf(
                pdf_id=f"{file_id}_chunk{chunk_idx}" if len(files_to_ocr) > 1 else file_id,
                pdf_path=chunk_path,
                api_key=MISTRAL_API_KEY,
                output_dir=str(DATA_OUTPUT),
                db_path=DB_PATH,
                model=MISTRAL_MODEL,
                delay_seconds=DELAY_SECONDS
            )

            all_results.append(result)
            total_api_calls += 1

            if result['success']:
                print(f"  ✓ OCR successful: {result['total_pages']} pages, {result['confidence']:.2f} confidence")
                if result['warnings']:
                    print(f"  ⚠️  Warnings: {len(result['warnings'])}")
            else:
                print(f"  ✗ Error: {result['error']}")
                total_errors += 1

        # 3. Summary for this file
        successful_chunks = sum(1 for r in all_results if r['success'])
        total_pages = sum(r.get('total_pages', 0) for r in all_results if r['success'])

        print(f"\n{'='*70}")
        if successful_chunks == len(all_results):
            print(f"✅ FILE COMPLETE: {file_info['filename']}")
            print(f"   Chunks: {len(all_results)}, Pages: {total_pages}")
            total_processed += 1
        else:
            print(f"⚠️  PARTIAL ERROR: {successful_chunks}/{len(all_results)} chunks successful")
            total_errors += 1
        print("="*70)

    except Exception as e:
        print(f"\n{'='*70}")
        print(f"✗ ERROR processing {file_info['filename']}: {e}")
        print("="*70)
        total_errors += 1

# Final statistics
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print("\n\n")
print("="*70)
print("BATCH PROCESSING COMPLETE")
print("="*70)
print("📊 Statistics:")
print(f"  Total files:        {total_files}")
print(f"  Successful:         {total_processed}")
print(f"  Errors:             {total_errors}")
print(f"  Already processed:  {len(processed_files)}")
print(f"  API calls:          {total_api_calls}")
print("")
print(f"⏱️  Processing time: {duration:.1f}s ({duration/60:.1f} min)")
print("")

# Detailed statistics from database
db_stats = get_processing_stats(DB_PATH)
print("📈 Database statistics:")
print(f"  Total processed:    {db_stats['completed']}")
print(f"  Total errors:       {db_stats['error']}")
print(f"  Avg. time:          {db_stats['avg_duration_sec']:.1f}s")
print(f"  Avg. confidence:    {db_stats['avg_confidence']:.2f}")
print(f"  Total pages:        {db_stats['total_pages']}")
print(f"  Estimated cost:     ${db_stats['estimated_cost_usd']:.4f}")
print("="*70)
print(f"✓ Results saved in: {DATA_OUTPUT}")
print("="*70)

BATCH OCR PROCESSING
📊 Status:
  Total files:        20
  Already processed:  256
  To do:              -236

🚀 Starting processing...


[1/20] 📄 Processing: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p008.jpg
  Type:   Image
  Size:   1.17 MB

📦 Preparing file...
✓ File ready (no splitting needed)

🔄 Starting OCR...


2026-03-16 12:33:43,831 - utils - INFO - Datei kodiert: 1221874 Bytes → 1629168 Zeichen Base64
2026-03-16 12:33:43,838 - utils - INFO - Verarbeite lokales Bild: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p008.jpg
2026-03-16 12:33:43,840 - utils - INFO - Starte OCR mit Modell: mistral-ocr-latest
2026-03-16 12:33:54,477 - httpx - INFO - HTTP Request: POST https://api.mistral.ai/v1/ocr "HTTP/1.1 200 OK"
2026-03-16 12:33:54,491 - utils - INFO - OCR erfolgreich: 1 Seiten, Modell: mistral-ocr-latest, 1193.2 KB, 10.78s
2026-03-16 12:33:54,499 - utils - INFO - OCR-Ergebnisse gespeichert: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p008
2026-03-16 12:33:54,507 - utils - INFO - ✓ PDF TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p008 verarbeitet: 1 Seiten, 525 Wörter, Confidence 0.80, 12.3s


  ✓ OCR successful: 1 pages, 0.80 confidence
  ⚠️  Warnings: 1

✅ FILE COMPLETE: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p008.jpg
   Chunks: 1, Pages: 1

[2/20] 📄 Processing: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p010.jpg
  Type:   Image
  Size:   1.15 MB

📦 Preparing file...
✓ File ready (no splitting needed)

🔄 Starting OCR...


2026-03-16 12:33:56,104 - utils - INFO - Datei kodiert: 1203300 Bytes → 1604400 Zeichen Base64
2026-03-16 12:33:56,105 - utils - INFO - Verarbeite lokales Bild: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p010.jpg
2026-03-16 12:33:56,107 - utils - INFO - Starte OCR mit Modell: mistral-ocr-latest
2026-03-16 12:34:00,408 - httpx - INFO - HTTP Request: POST https://api.mistral.ai/v1/ocr "HTTP/1.1 200 OK"
2026-03-16 12:34:00,419 - utils - INFO - OCR erfolgreich: 1 Seiten, Modell: mistral-ocr-latest, 1175.1 KB, 4.40s
2026-03-16 12:34:00,424 - utils - INFO - OCR-Ergebnisse gespeichert: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p010
2026-03-16 12:34:00,430 - utils - INFO - ✓ PDF TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p010 verarbeitet: 1 Seiten, 474 Wörter, Confidence 0.80, 5.9s


  ✓ OCR successful: 1 pages, 0.80 confidence
  ⚠️  Warnings: 1

✅ FILE COMPLETE: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p010.jpg
   Chunks: 1, Pages: 1

[3/20] 📄 Processing: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p020.jpg
  Type:   Image
  Size:   1.27 MB

📦 Preparing file...
✓ File ready (no splitting needed)

🔄 Starting OCR...


...

  ✓ OCR successful: 1 pages, 0.80 confidence
  ⚠️  Warnings: 1

✅ FILE COMPLETE: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p017.jpg
   Chunks: 1, Pages: 1

[19/20] 📄 Processing: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p016.jpg
  Type:   Image
  Size:   1.38 MB

📦 Preparing file...
✓ File ready (no splitting needed)

🔄 Starting OCR...


2026-03-16 12:35:26,719 - utils - INFO - Datei kodiert: 1442850 Bytes → 1923800 Zeichen Base64
2026-03-16 12:35:26,720 - utils - INFO - Verarbeite lokales Bild: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p016.jpg
2026-03-16 12:35:26,721 - utils - INFO - Starte OCR mit Modell: mistral-ocr-latest
2026-03-16 12:35:31,029 - httpx - INFO - HTTP Request: POST https://api.mistral.ai/v1/ocr "HTTP/1.1 200 OK"
2026-03-16 12:35:31,045 - utils - INFO - OCR erfolgreich: 1 Seiten, Modell: mistral-ocr-latest, 1409.0 KB, 4.43s
2026-03-16 12:35:31,054 - utils - INFO - OCR-Ergebnisse gespeichert: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p016
2026-03-16 12:35:31,068 - utils - INFO - ✓ PDF TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p016 verarbeitet: 1 Seiten, 513 Wörter, Confidence 0.80, 5.9s


  ✓ OCR successful: 1 pages, 0.80 confidence
  ⚠️  Warnings: 1

✅ FILE COMPLETE: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p016.jpg
   Chunks: 1, Pages: 1

[20/20] 📄 Processing: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p002.jpg
  Type:   Image
  Size:   1.5 MB

📦 Preparing file...
✓ File ready (no splitting needed)

🔄 Starting OCR...


2026-03-16 12:35:32,701 - utils - INFO - Datei kodiert: 1573077 Bytes → 2097436 Zeichen Base64
2026-03-16 12:35:32,702 - utils - INFO - Verarbeite lokales Bild: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p002.jpg
2026-03-16 12:35:32,703 - utils - INFO - Starte OCR mit Modell: mistral-ocr-latest
2026-03-16 12:35:36,770 - httpx - INFO - HTTP Request: POST https://api.mistral.ai/v1/ocr "HTTP/1.1 200 OK"
2026-03-16 12:35:36,777 - utils - INFO - OCR erfolgreich: 1 Seiten, Modell: mistral-ocr-latest, 1536.2 KB, 4.20s
2026-03-16 12:35:36,782 - utils - INFO - OCR-Ergebnisse gespeichert: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p002
2026-03-16 12:35:36,792 - utils - INFO - ✓ PDF TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p002 verarbeitet: 1 Seiten, 420 Wörter, Confidence 1.00, 5.7s


  ✓ OCR successful: 1 pages, 1.00 confidence

✅ FILE COMPLETE: TS_GER_1953_EXT-PUB_Wiebe-H_-_Siedlungswerk-Mennoniten-Weichseltal_SAMPLE-20_EVAL_p002.jpg
   Chunks: 1, Pages: 1



BATCH PROCESSING COMPLETE
📊 Statistics:
  Total files:        20
  Successful:         20
  Errors:             0
  Already processed:  256
  API calls:          20

⏱️  Processing time: 114.6s (1.9 min)

📈 Database statistics:
  Total processed:    276
  Total errors:       6
  Avg. time:          5.8s
  Avg. confidence:    0.77
  Total pages:        324
  Estimated cost:     $0.3240
✓ Results saved in: /home/laptop-office/code/pubs_mistral-ocr/ocr-zeitschriften-mistral-public/data/output


## Cell 4: Cleanup & Storage Management

Deletes temporary PDF chunk files from `data/tracking/pdf_chunks/` to free up disk space. Analyzes storage before/after and shows freed space.

**Preserved:** SQLite database (checkpoint system) | OCR results (`.md`, `.txt`, `.json`) | Original PDFs in `data/input/`

**Deleted:** Only temporary PDF chunks from the splitting process (Cell 3)

**This cell can be run independently** (only requires Cell 1 for imports) | **Dry run:** Set `dry_run=True` in `cleanup_temp_files()` for test without deletion

In [ ]:
# Minimal imports if Cell 1 not executed
if 'PROJECT_ROOT' not in globals():
    from pathlib import Path
    from utils import get_storage_stats, cleanup_temp_files
    PROJECT_ROOT = Path("..").resolve()
    DATA_TRACKING = PROJECT_ROOT / "data" / "tracking"

# Define temp directory
TEMP_DIR = DATA_TRACKING / "pdf_chunks"

# Storage statistics BEFORE cleanup
print("="*70)
print("STORAGE ANALYSIS")
print("="*70)

try:
    storage_before = get_storage_stats(str(PROJECT_ROOT))
    
    print("📊 Current sizes:")
    print("")
    print(f"  Input PDFs:       {storage_before['input']:>8.2f} MB")
    print(f"  Output files:     {storage_before['output']:>8.2f} MB")
    print(f"  Tracking DB:      {storage_before['database']:>8.2f} MB")
    print(f"  PDF chunks:       {storage_before['chunks']:>8.2f} MB  ⚠️")
    print(f"  {'-'*40}")
    print(f"  TOTAL:            {storage_before['total']:>8.2f} MB")
    print("")
    
except Exception as e:
    logger.error(f"Error in storage analysis: {e}")
    print("⚠️  Warning: Storage statistics could not be calculated")
    print(f"   Error: {e}")
    print("")
    print("   Cleanup will still be attempted...")
    print("")
    storage_before = None

# Perform cleanup (even if stats failed)
try:
    if storage_before is None or storage_before.get('chunks', 0) >= 1.0:
        if storage_before and storage_before['chunks'] >= 1.0:
            print(f"⚠️  PDF chunks occupy {storage_before['chunks']:.2f} MB storage space.")
        else:
            print("🔄 Starting cleanup (storage info not available)...")
        
        print("")
        print("="*70)
        print("START CLEANUP")
        print("="*70)
        print("")

        # Perform cleanup
        try:
            cleanup_result = cleanup_temp_files(
                temp_dir=str(TEMP_DIR),
                dry_run=False  # Set to True for test run
            )

            print("")
            print("="*70)
            print("CLEANUP COMPLETE")
            print("="*70)
            print("")
            print("📋 Statistics:")
            print(f"  Found chunks:       {cleanup_result['files_found']}")
            print(f"  Deleted chunks:     {cleanup_result['files_deleted']}")
            print(f"  Freed:              {cleanup_result['space_freed_mb']:.2f} MB")
            print("")

            if cleanup_result['deleted_files']:
                print("🗑️  Deleted files:")
                for filename in cleanup_result['deleted_files'][:10]:  # Show max 10
                    print(f"  - {filename}")
                if len(cleanup_result['deleted_files']) > 10:
                    print(f"  ... and {len(cleanup_result['deleted_files']) - 10} more")
                print("")

        except Exception as e:
            logger.error(f"Cleanup failed: {e}")
            print("")
            print("="*70)
            print("✗ CLEANUP FAILED")
            print("="*70)
            print("")
            print(f"Error: {e}")
            print("")
            print("Possible causes:")
            print("  - File access issues (permissions)")
            print("  - Cloud sync active")
            print("  - Files used by another process")
            print("")
            print("💡 Tip: Try again later or delete chunks manually:")
            print(f"   {TEMP_DIR}")
            print("="*70)

        # Storage statistics AFTER cleanup (best effort)
        if storage_before is not None:
            try:
                storage_after = get_storage_stats(str(PROJECT_ROOT))

                print("="*70)
                print("STORAGE AFTER CLEANUP")
                print("="*70)
                print("")
                print("📊 New sizes:")
                print("")
                print(f"  Input PDFs:       {storage_after['input']:>8.2f} MB")
                print(f"  Output files:     {storage_after['output']:>8.2f} MB")
                print(f"  Tracking DB:      {storage_after['database']:>8.2f} MB")
                print(f"  PDF chunks:       {storage_after['chunks']:>8.2f} MB  ✓")
                print(f"  {'-'*40}")
                print(f"  TOTAL:            {storage_after['total']:>8.2f} MB")
                print("")
                print(f"💾 Saved: {storage_before['total'] - storage_after['total']:.2f} MB")
                print("="*70)
                print("✓ Cleanup successful!")
                print("="*70)
            except Exception as e:
                logger.warning(f"After-statistics could not be calculated: {e}")
                print("")
                print("⚠️  After-statistics not available (cleanup completed though)")
                print("="*70)
    else:
        print("✓ No PDF chunks found. Cleanup not needed.")
        print("="*70)

except Exception as e:
    # Unexpected error in entire cleanup workflow
    logger.error(f"Unexpected error in cleanup workflow: {e}")
    print("")
    print("="*70)
    print("✗ ERROR IN CLEANUP WORKFLOW")
    print("="*70)
    print("")
    print(f"An unexpected error occurred: {e}")
    print("")
    print("Cleanup could not be performed.")
    print("The pipeline will continue to work normally.")
    print("="*70)

STORAGE ANALYSIS
📊 Current sizes:

  Input PDFs:          25.65 MB
  Output files:         0.11 MB
  Tracking DB:          0.31 MB
  PDF chunks:           0.00 MB  ⚠️
  ----------------------------------------
  TOTAL:               26.38 MB

✓ No PDF chunks found. Cleanup not needed.
